# Encoder–HistGB: cosine annealing and HistGB optimization

Keep the five-layer encoder architecture from `opt.ipynb` and shared preprocessing from
`features_selection.ipynb`. First optimize learning rate, weight decay, dropout and restart
period with the previous best HistGB parameters. Confirm the two best encoder configurations,
then optimize HistGB using embeddings from the winning encoder settings, passed directly
through `winning_params`.

Both stages use the same encoder-training procedure: select the epoch count using an internal
20% patient holdout (maximum 160 epochs), then refit fresh preprocessing/scalers and a fresh
encoder on all fold-training patients for that many epochs. Outer-validation patients train
neither model. This replaces the separate fixed-160-epoch HistGB optimization experiment.

Encoder search: 30 trials, five fixed patient folds and seed 42. Confirm the best two on the
same folds with seeds 42, 43 and 44 (15 runs each). Cache the winning encoder's embeddings on
those five folds with seed 42, then run 40 HistGB trials without retraining the encoder.
These reused folds provide tuning/stability scores, not an independent test estimate.


## 1. Reuse the shared baseline and set training options

Only the five setup cells from features_selection.ipynb are executed, not its validation or
feature-selection experiments. Paths work from the project root or either notebook folder.
Architecture comes from opt.ipynb; HistGB starts from its previously tuned parameters.
N_TRIALS controls encoder search; TOP_K=2 controls confirmation. HISTGB_N_TRIALS controls the final HistGB search.


In [1]:
import json
import os
from pathlib import Path
from copy import deepcopy
import optuna
import torch
from torch import nn
from torch.nn.utils.rnn import pad_sequence
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GroupKFold

PROJECT_DIR = next(p for p in (Path.cwd(), *Path.cwd().parents)
                   if (p / "exploration/features_selection.ipynb").is_file())
reference = json.loads((PROJECT_DIR / "exploration/features_selection.ipynb").read_text())
previous_dir = Path.cwd()
try:
    os.chdir(PROJECT_DIR)
    for cell_id in ["selection-imports", "selection-history", "selection-imputation",
                    "selection-features", "selection-pipeline"]:
        exec("".join(next(c for c in reference["cells"] if c.get("id") == cell_id)["source"]))
finally:
    os.chdir(previous_dir)
DATA_DIR = PROJECT_DIR / "data"
baseline = clone(pipeline)

SEED = 42
CONFIRMATION_SEEDS = [42, 43, 44]
N_TRIALS, TOP_K = 30, 2
HISTGB_N_TRIALS = 40
# Saved confirmed encoder winner from opt.ipynb.
ENCODER = dict(width=32, heads=8, layers=5, dropout=0.1347886862599206,
               ff_multiplier=3, mlp_layers=2, mlp_units=16)
MAX_EPOCHS, PATIENCE, BATCH_SIZE = 160, 25, 128
LEARNING_RATE, WEIGHT_DECAY = 0.001759366313629433, 4.3514588969246546e-05
MIN_LEARNING_RATE = LEARNING_RATE * 0.01
RESTART_PERIOD, RESTART_MULTIPLIER = 20, 1  # Constant 20-epoch cosine cycles.
# Previously tuned HistGB parameters; used for both comparisons.
HISTGB_PARAMS = {
    "learning_rate": 0.03471523676107423, "max_iter": 800,
    "max_leaf_nodes": 62, "min_samples_leaf": 16,
    "l2_regularization": 8.305016398991823, "max_features": 0.6408427464001498,
}
baseline[-1].set_params(random_state=SEED, **HISTGB_PARAMS)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else
                      "mps" if torch.backends.mps.is_available() else "cpu")
torch.set_num_threads(3)
print(f"Device: {DEVICE}; baseline: {baseline[-1]}")

/Users/Luc/Library/CloudStorage/OneDrive-Personal/Documents/Cours/MS2A/Apprentissage Statistique/Méthodologie/Challenge_Parkinson_disease/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Device: mps; baseline: HistGradientBoostingRegressor(l2_regularization=8.305016398991823,
                              learning_rate=0.03471523676107423,
                              max_features=0.6408427464001498, max_iter=800,
                              max_leaf_nodes=62, min_samples_leaf=16,
                              random_state=42)


## 2. One sequence per patient

Patient IDs only group visits; they are never encoder inputs. Elapsed time encodes the actual
spacing between visits. Padding uses row number -1, which is ignored by attention and MSE.
The row numbers also restore embeddings to the original visit order after batching.

In [2]:
def encoder_features(imputed):
    features = imputed.drop(columns="patient_id").copy()
    features["time_since_first_visit"] = (
        imputed["age"] - imputed.groupby("patient_id")["age"].transform("min")
    )
    return features


def patient_sequences(raw, features, targets=None):
    visits = raw.reset_index(drop=True).sort_values(["patient_id", "age"])
    rows = [torch.tensor(group.index.to_numpy())
            for _, group in visits.groupby("patient_id", sort=False)]
    values = torch.tensor(np.asarray(features), dtype=torch.float32)
    labels = torch.tensor(np.zeros(len(raw)) if targets is None else np.asarray(targets),
                          dtype=torch.float32)
    return TensorDataset(
        pad_sequence([values[i] for i in rows], batch_first=True),
        pad_sequence([labels[i] for i in rows], batch_first=True),
        pad_sequence(rows, batch_first=True, padding_value=-1),
    )

## 3. Optimized bidirectional encoder

Dense input projection → five TransformerEncoder layers → one embedding per visit.
Use the same parameterized architecture as opt.ipynb: width 32, eight heads, feed-forward
multiplier three, and a regression head with two hidden layers of 16 units. The regression
head supervises training; HistGB receives the contextual embeddings.


In [3]:
class VisitEncoder(nn.Module):
    def __init__(self, n_features, width=64, heads=4, layers=2, dropout=0.1,
                 ff_multiplier=2, mlp_layers=0, mlp_units=32):
        super().__init__()
        self.projection = nn.Linear(n_features, width)
        layer = nn.TransformerEncoderLayer(
            d_model=width, nhead=heads, dim_feedforward=ff_multiplier * width,
            dropout=dropout, activation="gelu", batch_first=True,
        )
        self.encoder = nn.TransformerEncoder(layer, num_layers=layers,
                                             enable_nested_tensor=False)
        # 1. Initialize the encoder’s weights
        # TransformerEncoder copies its layer: give each block independent weights.
        for parameter in self.encoder.parameters():
            if parameter.ndim > 1:
                nn.init.xavier_uniform_(parameter)
        # 2. Build the regression head
        head, input_dim = [], width
        for _ in range(mlp_layers):
            head.extend([nn.Linear(input_dim, mlp_units), nn.GELU(), nn.Dropout(dropout)])
            input_dim = mlp_units
        self.head = nn.Sequential(*head, nn.Linear(input_dim, 1))

    def encode(self, values, padding):
        return self.encoder(self.projection(values), src_key_padding_mask=padding,
                            is_causal=False)

    def forward(self, values, padding):
        return self.head(self.encode(values, padding)).squeeze(-1)

## 4. Parameterized cosine annealing and internal early stopping

Each trial tests a different learning rate, weight decay, dropout, and restart period. The encoder architecture stays fixed at five layers. Cosine cycles keep the same length within one training, and the minimum learning rate is always 1% of the trial’s starting value.

Keep the 160-epoch ceiling. Patience is max(25, 2 × restart period), allowing at least two
cycles without improvement before stopping. Select the epoch with the lowest internal
holdout MSE, then refit preprocessing/scalers and a fresh encoder on all fold-training
patients for exactly that many epochs. Outer-validation targets never select the epoch.


In [4]:
def train_network(training, seed, epochs, validation=None, params=None):
    params = {"learning_rate": LEARNING_RATE, "weight_decay": WEIGHT_DECAY,
              "dropout": ENCODER["dropout"], "restart_period": RESTART_PERIOD, **(params or {})}
    patience = max(PATIENCE, 2 * params["restart_period"])
    torch.manual_seed(seed)
    network = VisitEncoder(training.tensors[0].shape[-1], **{**ENCODER, "dropout": params["dropout"]}).to(DEVICE)
    optimizer = torch.optim.AdamW(network.parameters(), lr=params["learning_rate"],
                                 weight_decay=params["weight_decay"])
    scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
        optimizer, T_0=params["restart_period"], T_mult=RESTART_MULTIPLIER,
        eta_min=params["learning_rate"] * 0.01,
    )
    loader = DataLoader(training, batch_size=BATCH_SIZE, shuffle=True)
    best_loss, best_epoch, best_state, stale = np.inf, epochs, None, 0
    history = []

    for epoch in range(1, epochs + 1):
        network.train()
        total_error, count = 0.0, 0
        for values, targets, rows in loader:
            values, targets, rows = (t.to(DEVICE) for t in (values, targets, rows))
            valid = rows >= 0
            optimizer.zero_grad()
            prediction = network(values, ~valid)
            loss = nn.functional.mse_loss(prediction[valid], targets[valid])
            loss.backward()
            nn.utils.clip_grad_norm_(network.parameters(), 1.0)
            optimizer.step()
            total_error += loss.item() * valid.sum().item()
            count += valid.sum().item()
        history.append({"epoch": epoch, "train_MSE": total_error / count,
                        "learning_rate": optimizer.param_groups[0]["lr"]})

        if validation is not None:
            network.eval()
            total_error, count = 0.0, 0
            with torch.no_grad():
                for values, targets, rows in DataLoader(validation, batch_size=BATCH_SIZE):
                    values, targets, rows = (t.to(DEVICE) for t in (values, targets, rows))
                    valid = rows >= 0
                    prediction = network(values, ~valid)
                    total_error += ((prediction[valid] - targets[valid]) ** 2).sum().item()
                    count += valid.sum().item()
            stop_loss = total_error / count
            history[-1]["stop_MSE"] = stop_loss
            if stop_loss < best_loss:
                best_loss, best_epoch, stale = stop_loss, epoch, 0
                best_state = deepcopy(network.state_dict())
            else:
                stale += 1
            if stale >= patience:
                break
        scheduler.step()

    if best_state is not None:
        network.load_state_dict(best_state)
    return network.eval(), best_epoch, pd.DataFrame(history)


def fit_encoder(raw, targets, seed, params=None):
    fit_idx, stop_idx = next(GroupShuffleSplit(
        n_splits=1, test_size=0.2, random_state=seed
    ).split(raw, groups=raw["patient_id"]))
    fit_raw, stop_raw = raw.iloc[fit_idx], raw.iloc[stop_idx]
    inner_prep = clone(baseline[:2])
    fit_features = encoder_features(inner_prep.fit_transform(fit_raw))
    stop_features = encoder_features(inner_prep.transform(stop_raw))
    scaler = StandardScaler().fit(fit_features)
    mean = targets.iloc[fit_idx].mean()
    scale = max(targets.iloc[fit_idx].std(ddof=0), 1e-8)
    training = patient_sequences(fit_raw, scaler.transform(fit_features),
                                 (targets.iloc[fit_idx] - mean) / scale)
    stopping = patient_sequences(stop_raw, scaler.transform(stop_features),
                                 (targets.iloc[stop_idx] - mean) / scale)
    _, epochs, history = train_network(training, seed, MAX_EPOCHS, stopping, params=params)
    history[["train_MSE", "stop_MSE"]] *= scale ** 2

    # Fresh fit on all fold-training patients, using the selected fixed epoch count.
    full_prep = clone(baseline[:2])
    features = encoder_features(full_prep.fit_transform(raw))
    scaler = StandardScaler().fit(features)
    mean, scale = targets.mean(), max(targets.std(ddof=0), 1e-8)
    training = patient_sequences(raw, scaler.transform(features), (targets - mean) / scale)
    network, _, refit_history = train_network(training, seed, epochs, validation=None, params=params)
    refit_history["train_MSE"] *= scale ** 2

    network.requires_grad_(False)
    return dict(network=network, prep=full_prep, scaler=scaler,
                target_mean=mean, target_scale=scale,
                epochs=epochs, history=history, refit_history=refit_history)

## 5. Extract frozen embeddings to feed them to histgb

Creates a function to retrieve the embeddings of the encoder to give them to  HistGB. To do so we simply pass the encoder input features through the encoder.

In [5]:
@torch.no_grad()
def encoder_outputs(fitted, raw):
    imputed = fitted["prep"].transform(raw)
    features = fitted["scaler"].transform(encoder_features(imputed))
    sequences = patient_sequences(raw, features)
    network = fitted["network"].eval()
    embeddings = np.empty((len(raw), network.projection.out_features), dtype=np.float32)
    predictions = np.empty(len(raw), dtype=np.float32)
    for values, _, rows in DataLoader(sequences, batch_size=BATCH_SIZE):
        padding = rows < 0
        hidden = network.encode(values.to(DEVICE), padding.to(DEVICE))
        prediction = network.head(hidden).squeeze(-1).cpu()
        valid_rows = rows[~padding].numpy()
        embeddings[valid_rows] = hidden.cpu()[~padding].numpy()
        predictions[valid_rows] = prediction[~padding].numpy()
    embeddings = pd.DataFrame(embeddings, index=raw.index,
                              columns=[f"enc_{i:02d}" for i in range(embeddings.shape[1])])
    predictions = predictions * fitted["target_scale"] + fitted["target_mean"]
    return embeddings, pd.Series(predictions, index=raw.index)

## 6.1 Prepare five patient folds and create a function to train the encoder + histgb on the five folds and measure the perf


In [6]:
cv = GroupKFold(n_splits=5, shuffle=True, random_state=SEED)
prepared_folds = []
with threadpool_limits(limits=3):
    for train_idx, val_idx in cv.split(X, y, groups=X["patient_id"]):
        raw_train, raw_val = X.iloc[train_idx], X.iloc[val_idx]
        yt, yv = y.iloc[train_idx], y.iloc[val_idx]
        prep = clone(baseline[:-1])
        prepared_folds.append(dict(
            raw_train=raw_train, raw_val=raw_val, yt=yt, yv=yv,
            train=prep.fit_transform(raw_train, yt), validation=prep.transform(raw_val),
        ))


def evaluate_config(params, folds, seeds, trial=None):
    scores = []
    with threadpool_limits(limits=3):
        for fold_number, fold in enumerate(folds, 1):
            for seed in seeds:
                fitted = fit_encoder(fold["raw_train"], fold["yt"], seed, params=params)
                z_train, _ = encoder_outputs(fitted, fold["raw_train"])
                z_val, _ = encoder_outputs(fitted, fold["raw_val"])
                model = clone(baseline[-1]).set_params(random_state=seed)
                model.fit(fold["train"].join(z_train), fold["yt"])
                mse = mean_squared_error(fold["yv"], model.predict(fold["validation"].join(z_val)))
                scores.append(dict(fold=fold_number, seed=seed, epochs=fitted["epochs"], MSE=mse))
                print(f"Fold {fold_number}/5, seed {seed}, {fitted['epochs']} epochs: MSE {mse:.4f}")
            if trial is not None:
                trial.report(float(np.mean([s["MSE"] for s in scores])), step=fold_number)
                if trial.should_prune():
                    raise optuna.TrialPruned()
    return pd.DataFrame(scores)


## 6.2 Optuna search on Encoder parameters: five folds, one seed

Search 30 configurations by default, starting with the existing settings. Learning rate:
1e-4–5e-3; weight decay: 1e-6–1e-2 (both logarithmic); dropout: 0–0.35; restart period:
10, 15, 20, 30 or 40 epochs. Minimize complete Encoder–HistGB MSE. Median pruning can
stop trials after at least two folds, following five startup trials.


In [ ]:
def objective(trial):
    params = dict(
        learning_rate=trial.suggest_float("learning_rate", 1e-4, 5e-3, log=True),
        weight_decay=trial.suggest_float("weight_decay", 1e-6, 1e-2, log=True),
        dropout=trial.suggest_float("dropout", 0.0, 0.35),
        restart_period=trial.suggest_categorical("restart_period", [10, 15, 20, 30, 40]),
    )
    result = evaluate_config(params, prepared_folds, [SEED], trial)
    trial.set_user_attr("fold_scores", result.to_dict("records"))
    return float(result["MSE"].mean())


study = optuna.create_study(
    direction="minimize", sampler=optuna.samplers.TPESampler(seed=SEED),
    pruner=optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=2),
)
study.enqueue_trial(dict(learning_rate=LEARNING_RATE, weight_decay=WEIGHT_DECAY,
                         dropout=ENCODER["dropout"], restart_period=RESTART_PERIOD))
study.optimize(objective, n_trials=N_TRIALS, gc_after_trial=True, show_progress_bar=True)
study.trials_dataframe().sort_values("value")


[I 2026-11-05 16:10:34,033] A new study created in memory with name: no-name-c38890d0-b88b-4f68-a042-d060adef107b
  0%|          | 0/30 [00:00<?, ?it/s]

Fold 1/5, seed 42, 139 epochs: MSE 6.9049
Fold 2/5, seed 42, 153 epochs: MSE 7.0431
Fold 3/5, seed 42, 160 epochs: MSE 7.1497
Fold 4/5, seed 42, 160 epochs: MSE 7.5876


Best trial: 0. Best value: 7.09491:   3%|▎         | 1/30 [26:51<12:59:07, 1611.97s/it]

Fold 5/5, seed 42, 159 epochs: MSE 6.7892
[I 2026-11-05 16:37:25,878] Trial 0 finished with value: 7.094911319717748 and parameters: {'learning_rate': 0.001759366313629433, 'weight_decay': 4.3514588969246546e-05, 'dropout': 0.1347886862599206, 'restart_period': 20}. Best is trial 0 with value: 7.094911319717748.
Fold 1/5, seed 42, 155 epochs: MSE 8.0662
Fold 2/5, seed 42, 147 epochs: MSE 8.1815
Fold 3/5, seed 42, 149 epochs: MSE 8.3299
Fold 4/5, seed 42, 155 epochs: MSE 9.3170


Best trial: 0. Best value: 7.09491:   7%|▋         | 2/30 [51:42<11:58:52, 1540.45s/it]

Fold 5/5, seed 42, 154 epochs: MSE 8.2483
[I 2026-11-05 17:02:16,320] Trial 1 finished with value: 8.42859915856736 and parameters: {'learning_rate': 0.00043284502212938834, 'weight_decay': 0.006351221010640699, 'dropout': 0.25619787963399177, 'restart_period': 40}. Best is trial 0 with value: 7.094911319717748.
Fold 1/5, seed 42, 59 epochs: MSE 8.1946
Fold 2/5, seed 42, 60 epochs: MSE 8.3076
Fold 3/5, seed 42, 69 epochs: MSE 8.2276
Fold 4/5, seed 42, 60 epochs: MSE 8.9485


... (8 outputs left out)

Best trial: 0. Best value: 7.09491:  23%|██▎       | 7/30 [2:05:01<5:38:32, 883.14s/it] 

Fold 2/5, seed 42, 157 epochs: MSE 8.3612
[I 2026-11-05 18:15:35,417] Trial 6 pruned. 
Fold 1/5, seed 42, 158 epochs: MSE 7.7336
Fold 2/5, seed 42, 159 epochs: MSE 7.9743
Fold 3/5, seed 42, 160 epochs: MSE 7.9236
Fold 4/5, seed 42, 159 epochs: MSE 8.0537


Best trial: 0. Best value: 7.09491:  27%|██▋       | 8/30 [2:25:28<6:03:55, 992.51s/it]

Fold 5/5, seed 42, 150 epochs: MSE 8.0006
[I 2026-11-05 18:36:02,079] Trial 7 finished with value: 7.937164160302446 and parameters: {'learning_rate': 0.000848876216140871, 'weight_decay': 5.4880470007660465e-06, 'dropout': 0.33935461971759545, 'restart_period': 15}. Best is trial 0 with value: 7.094911319717748.
Fold 1/5, seed 42, 158 epochs: MSE 8.6689


## 6.3 Confirm the best two: five folds, three seeds

Retrain each of the two best completed configurations on all five folds with seeds 42, 43
and 44. Each seed controls encoder initialization, the internal patient holdout and HistGB.
Select the epoch count on the internal holdout, then refit on all fold-training patients. The winning
configuration minimizes mean MSE across its 15 runs. Standard deviations describe
fold/seed variation; the patients and validation folds are reused from model selection.


In [ ]:
candidates = sorted(
    (t for t in study.trials if t.state == optuna.trial.TrialState.COMPLETE),
    key=lambda t: t.value,
)[:TOP_K]
if len(candidates) < TOP_K:
    raise ValueError("At least two completed trials are required for confirmation.")

confirmation = []
for candidate in candidates:
    print(f"Confirming trial {candidate.number}: {candidate.params}")
    result = evaluate_config(candidate.params, prepared_folds, CONFIRMATION_SEEDS)
    confirmation.append(result.assign(trial=candidate.number))
confirmation = pd.concat(confirmation, ignore_index=True)
confirmation_summary = confirmation.groupby("trial").agg(
    MSE=("MSE", "mean"), std=("MSE", lambda s: s.std(ddof=0)),
    mean_epochs=("epochs", "mean"),
).sort_values("MSE")
confirmation_summary["RMSE"] = np.sqrt(confirmation_summary["MSE"])
winning_trial = int(confirmation_summary.index[0])
winning_params = next(t.params for t in candidates if t.number == winning_trial)
print("Winning parameters:", winning_params)
confirmation_summary


## 7. Cache the winning encoder embeddings


In [ ]:
# 7. Cache the winning encoder embeddings on the same five folds, seed 42.
tuning_folds = []
with threadpool_limits(limits=3):
    for fold_number, fold in enumerate(prepared_folds, 1):
        fitted_encoder = fit_encoder(fold["raw_train"], fold["yt"], SEED, params=winning_params)
        z_train, _ = encoder_outputs(fitted_encoder, fold["raw_train"])
        z_val, _ = encoder_outputs(fitted_encoder, fold["raw_val"])
        tuning_folds.append((fold["train"].join(z_train), fold["validation"].join(z_val),
                             fold["yt"], fold["yv"]))
        print(f"Fold {fold_number}/5: winning encoder refitted for "
              f"{fitted_encoder['epochs']} selected epochs; embeddings cached.")


## 8. Optimize HistGB with the winning encoder


In [ ]:
# 8. Optimize HistGB only; reuse cached embeddings for every trial.
# Reuse the fixed encoder embeddings from five patient folds and one seed.
def histgb_objective(trial):
    params = {
        "learning_rate": trial.suggest_float("learning_rate", 0.02, 0.2, log=True),
        "max_iter": trial.suggest_int("max_iter", 100, 800, step=100),
        "max_leaf_nodes": trial.suggest_int("max_leaf_nodes", 7, 63, log=True),
        "min_samples_leaf": trial.suggest_int("min_samples_leaf", 10, 120, log=True),
        "l2_regularization": trial.suggest_float("l2_regularization", 0, 30),
        "max_features": trial.suggest_float("max_features", 0.6, 1.0),
    }
    scores = []
    for step, (train, validation, yt, yv) in enumerate(tuning_folds):
        fitted = clone(baseline[-1]).set_params(**params).fit(train, yt)
        scores.append(mean_squared_error(yv, fitted.predict(validation)))
        # Report each patient fold; prune after at least two folds.
        trial.report(float(np.mean(scores)), step)
        if 1 <= step < len(tuning_folds) - 1 and trial.should_prune():
            raise optuna.TrialPruned()
    trial.set_user_attr("fold_mse", scores)
    return float(np.mean(scores))


histgb_study = optuna.create_study(
    direction="minimize", sampler=optuna.samplers.TPESampler(seed=SEED),
    pruner=optuna.pruners.MedianPruner(n_startup_trials=8, n_warmup_steps=1),
)
parameter_names = ["learning_rate", "max_iter", "max_leaf_nodes", "min_samples_leaf", "l2_regularization", "max_features"]
histgb_study.enqueue_trial({name: baseline[-1].get_params()[name] for name in parameter_names})
with threadpool_limits(limits=3):
    histgb_study.optimize(histgb_objective, n_trials=HISTGB_N_TRIALS)

baseline_scores = histgb_study.trials[0].user_attrs["fold_mse"]
best_scores = histgb_study.best_trial.user_attrs["fold_mse"]
tuning_summary = pd.DataFrame([
    {"model": name, "MSE": np.mean(scores), "std": np.std(scores)}
    for name, scores in [("Encoder_HistGB_current", baseline_scores), ("Encoder_HistGB_tuned", best_scores)]
]).set_index("model")
tuning_summary["RMSE"] = np.sqrt(tuning_summary["MSE"])
tuning_summary["gain_MSE_pct"] = 100 * (1 - tuning_summary["MSE"] / np.mean(baseline_scores))
best_histgb = clone(baseline[-1]).set_params(**histgb_study.best_params)
print("Best HistGB parameters:", histgb_study.best_params)
tuning_summary
